# Proposed Profile-Conditioned Semantic Retrieval

This notebook implements the proposed query-conditioned recommendation method.

The method extends the BLAIR-style semantic retrieval baseline by incorporating
the user's historical preference profile into the query representation.

For each evaluation example, the input to the semantic encoder is:

User Preference Profile + Synthetic Query

The combined text is encoded once using E5-base-v2 as a query representation.
The resulting 768-dimensional embedding is then searched against the same exact
FAISS index containing all products in the Amazon Electronics catalogue.

The proposed pipeline is:

User Preference Profile
        +
Synthetic Query
        ↓
Combined Query Representation
        ↓
E5 Query Embedding
        ↓
Exact FAISS Search
        ↓
Top-K Product Recommendations

The product encoder, product catalogue, FAISS index, retrieval depth, and
evaluation examples are identical to those used by the BLAIR-style baseline.

The experimental difference is therefore the inclusion of the user preference
profile in the query representation.

### Live Demonstration — Profile-Conditioned Semantic Retrieval

The proposed approach combines the user's historical preference profile with the current natural-language query before semantic encoding and retrieval.

In [1]:
import pandas as pd

# Load the saved Profile + Query (512) retrieval results
profile_query_512 = pd.read_parquet(
    "/users/adhp511/query-conditioned-recommender/data/generated/proposed_ablation_profile_query_512.parquet"
)

# Load the full product catalogue
catalogue_df = pd.read_parquet(
    "/users/adhp511/query-conditioned-recommender/data/generated/full_product_catalogue.parquet"
)

print(f"Profile + Query examples: {len(profile_query_512):,}")
print(f"Catalogue products: {len(catalogue_df):,}")

Profile + Query examples: 5,559
Catalogue products: 1,610,012


In [2]:
demo_sample_id = 1383

proposed_demo = profile_query_512[
    profile_query_512["sample_id"] == demo_sample_id
].iloc[0]

retrieved_asins = proposed_demo["retrieved_asins"]
retrieval_scores = proposed_demo["retrieval_scores"]

print("PROFILE + QUERY — TOP 10 RESULTS")
print("=" * 80)

for rank, (asin, score) in enumerate(
    zip(retrieved_asins[:10], retrieval_scores[:10]), start=1
):
    product = catalogue_df[
        catalogue_df["parent_asin"] == asin
    ].iloc[0]

    print(f"{rank:2d}. {product['title']}")
    print(f"    ASIN: {asin} | Score: {score:.4f}")

PROFILE + QUERY — TOP 10 RESULTS
 1. Yifantacy Power Trend MFI Certified 2M 6.6FT Nylon Braided USB to Lightning Cable Data Charging Cord for iPhone 5 / 5C / 5S / SE / 6 / 6S / 7/7 Plus/iPad/iPod Nano 7 (Luxury Gold)
    ASIN: B071HGV9GX | Score: 0.8451
 2. JSAUX 【Apple MFi Certified】 Lightning Cable 10ft, iPhone Charger Cable Nylon Braided Heavy Duty, Upgraded C89 USB Lightning Cord for iPhone 11 Xs Max X XR 8 7 6s 6 Plus SE 5 5s, iPad, iPod-Red
    ASIN: B082X666WZ | Score: 0.8446
 3. JSAUX Lightning iPhone Charger Cable, [Apple MFi Certified] 6ft Nylon Braided USB Fast Charging Cable Cord Compatible with iPhone 11 XS Max X XR 8 7 6s 6 Plus SE 5 5s 5c, iPad, iPod - White
    ASIN: B07MLV7TRJ | Score: 0.8442
 4. iPhone Lightning Cable - ilikable 3 Pack 6 FT MFi Certified Nylon Braided Charging Cable Compatible with iPhone 13 12 Pro MaxSE(2020) 11 Xs Max XR X 8 Plus 7/7 Plus / 6/6 Plus SE iPad Pro,Blue&White
    ASIN: B08BHPCNVF | Score: 0.8440
 5. JSAUX Lightning iPhone Charger Cable 

In [4]:
import numpy as np

ground_truth = proposed_demo["ground_truth_asin"]

matches = np.where(retrieved_asins == ground_truth)[0]

print("\n" + "=" * 80)

if len(matches) == 0:
    print("GROUND-TRUTH RESULT: NOT IN TOP-50")
else:
    rank = matches[0] + 1
    print(f"GROUND-TRUTH RESULT: RANK {rank}")
    print(f"GROUND-TRUTH ASIN: {ground_truth}")


GROUND-TRUTH RESULT: RANK 2
GROUND-TRUTH ASIN: B082X666WZ


In [5]:
print("RETRIEVAL COMPARISON")
print("=" * 80)

print("QUERY-ONLY BASELINE")
print("-" * 80)
print("Ground-truth rank: NOT IN TOP-50")

print("\nPROFILE + QUERY")
print("-" * 80)
print("Ground-truth rank: #2")

print("\n" + "=" * 80)
print("RESULT:")
print("The profile-conditioned model retrieves the exact ground-truth")
print("product at rank #2, whereas the query-only baseline does not")
print("retrieve it within the Top-50.")

RETRIEVAL COMPARISON
QUERY-ONLY BASELINE
--------------------------------------------------------------------------------
Ground-truth rank: NOT IN TOP-50

PROFILE + QUERY
--------------------------------------------------------------------------------
Ground-truth rank: #2

RESULT:
The profile-conditioned model retrieves the exact ground-truth
product at rank #2, whereas the query-only baseline does not
retrieve it within the Top-50.


## 1. Setup and Shared Retrieval Infrastructure

We reuse the exact semantic encoder and FAISS product index constructed in
`08_faiss_semantic_retrieval.ipynb`.

The proposed method does not modify the product representations or FAISS index.
Only the query-side representation is changed by incorporating the user's
preference profile alongside the synthetic query.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import faiss

# Project paths
PROJECT_ROOT = Path("/users/adhp511/query-conditioned-recommender")

DATA_DIR = PROJECT_ROOT / "data"
GENERATED_DIR = DATA_DIR / "generated"

EVAL_DATA_PATH = GENERATED_DIR / "proposed_model_data.parquet"
PRODUCT_CATALOGUE_PATH = GENERATED_DIR / "full_product_catalogue.parquet"
EMBEDDINGS_PATH = GENERATED_DIR / "e5_product_embeddings.npy"
FAISS_INDEX_PATH = GENERATED_DIR / "e5_faiss_product_index.bin"

# Local E5 model
E5_MODEL_PATH = Path.home() / "models" / "e5-base-v2"

# Retrieval settings
BATCH_SIZE = 256
MAX_LENGTH = 256
TOP_K = 50

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("E5 model:", E5_MODEL_PATH)
print("Evaluation data:", EVAL_DATA_PATH)
print("FAISS index:", FAISS_INDEX_PATH)

/users/adhp511/.pyenv/versions/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
E5 model: /users/adhp511/models/e5-base-v2
Evaluation data: /users/adhp511/query-conditioned-recommender/data/generated/proposed_model_data.parquet
FAISS index: /users/adhp511/query-conditioned-recommender/data/generated/e5_faiss_product_index.bin


## 2. Load the E5 Semantic Encoder

We use the same locally stored `E5-base-v2` model as the BLAIR-style
baseline.

E5 is used as the shared semantic encoder for both query representations
and product representations. In this notebook, we only need to encode the
profile-conditioned queries because the product embeddings have already
been generated and stored in the shared retrieval infrastructure.

In [2]:
# Load E5 tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_PATH)
e5_model = AutoModel.from_pretrained(E5_MODEL_PATH)

e5_model = e5_model.to(DEVICE)
e5_model.eval()

print("E5 model loaded successfully.")
print("Device:", DEVICE)
print("Hidden size:", e5_model.config.hidden_size)

Loading weights: 100%|██████████████████████████████████████████████| 199/199 [00:00<00:00, 8439.58it/s]


E5 model loaded successfully.
Device: cuda
Hidden size: 768


## 3. Load Evaluation Examples

Each evaluation example contains:

- `sample_id` — unique evaluation example
- `user_id` — user associated with the example
- `profile_text` — historical user preference profile
- `synthetic_query` — target-aware synthetic user query
- `ground_truth_asin` — the held-out product that should be retrieved

The profile and synthetic query will be combined into a single query-side
text representation. The ground-truth product is used only for evaluation.

In [3]:
# Load only the columns required for the proposed retrieval experiment
eval_data = pd.read_parquet(
    EVAL_DATA_PATH,
    columns=[
        "sample_id",
        "user_id",
        "synthetic_query",
        "profile_text",
        "ground_truth_asin",
    ],
)

print("Shape:", eval_data.shape)
print("\nColumns:")
print(eval_data.columns.tolist())

print("\nMissing values:")
print(eval_data.isna().sum())

print("\nUnique sample IDs:", eval_data["sample_id"].nunique())
print("Unique users:", eval_data["user_id"].nunique())
print("Unique ground-truth products:", eval_data["ground_truth_asin"].nunique())

eval_data.head()

Shape: (5559, 5)

Columns:
['sample_id', 'user_id', 'synthetic_query', 'profile_text', 'ground_truth_asin']

Missing values:
sample_id            0
user_id              0
synthetic_query      0
profile_text         0
ground_truth_asin    0
dtype: int64

Unique sample IDs: 5559
Unique users: 390
Unique ground-truth products: 5016


,sample_id,user_id,synthetic_query,profile_text,ground_truth_asin
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,Looking for a high-speed micro sd card for my ...,User Preference Profile\n\nThe user has primar...,B007VKII6A
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,looking for a good quality internal hard drive...,User Preference Profile\n\nThe user has primar...,B07454F4JH
2,2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,looking for a good quality internal hard drive...,User Preference Profile\n\nThe user has primar...,B00SMVLPIK
3,3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,looking for a good quality battery pack for my...,User Preference Profile\n\nThe user has primar...,B0179DDIQI
4,4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,looking for good quality on-ear headphones wit...,User Preference Profile\n\nThe user has primar...,B00X088DZI


## 4. Construct Profile-Conditioned Query Representations

For the proposed method, the user's historical preference profile and the
synthetic query are concatenated into a single text input.

The combined text is then encoded **once** by E5 as a query representation.

We do not separately encode the profile and query, and we do not average
their embeddings. The semantic encoder receives the complete combined text.

In [4]:
def build_profile_conditioned_query(profile, query):
    profile = str(profile).strip()
    query = str(query).strip()

    return (
        "User Preference Profile:\n"
        f"{profile}\n\n"
        "User Query:\n"
        f"{query}"
    )


# Construct the proposed query representation for every evaluation example
eval_data["profile_conditioned_query"] = [
    build_profile_conditioned_query(profile, query)
    for profile, query in zip(
        eval_data["profile_text"],
        eval_data["synthetic_query"]
    )
]

print("Example profile-conditioned query:\n")
print(eval_data.loc[0, "profile_conditioned_query"])

print("\nCharacter length:",
      len(eval_data.loc[0, "profile_conditioned_query"]))

Example profile-conditioned query:

User Preference Profile:
User Preference Profile

The user has primarily purchased products in these categories:

- Micro SD Cards
- Portable Bluetooth Speakers
- Internal Hard Drives

Products the user rated highly:

- Patriot LX Series 64GB High Speed Micro SDXC Class 10 UHS-I Transfer Speeds For Action Cameras, Phones, Tablets, and PCs
- The AquaAudio Cubo – Waterproof Bluetooth Wireless Speaker with Strong Suction Cup
- Western Digital 2TB WD Black Performance Internal Hard Drive HDD - 7200 RPM, SATA 6 Gb/s, 64 MB Cache, 3.5" - WD2003FZEX
- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-Tower

Most recent purchases:

- EBL Pack of 6 1000mAh NI-MH Home Phone Rechargeable Battery for HHRP105 HHR-P105 HHRP105A, HHR-P105A KX242
- CyberPower CP900AVR AVR UPS System, 900VA/560W, 10 Outlets, Mini-Tower

The user is generally hig

## 5. Validate Profile-Conditioned Inputs

Before encoding, we verify that every evaluation example contains a valid
user profile and synthetic query, and that the combined representation has
been constructed without missing values.

In [5]:
# Validate required fields
print("Total examples:", len(eval_data))

print("\nMissing profile-conditioned queries:",
      eval_data["profile_conditioned_query"].isna().sum())

print("Empty profile-conditioned queries:",
      (eval_data["profile_conditioned_query"].str.strip() == "").sum())

print("Missing profiles:",
      eval_data["profile_text"].isna().sum())

print("Missing synthetic queries:",
      eval_data["synthetic_query"].isna().sum())

print("\nProfile-conditioned query length statistics:")
print(
    eval_data["profile_conditioned_query"]
    .str.split()
    .str.len()
    .describe()
)

Total examples: 5559

Missing profile-conditioned queries: 0
Empty profile-conditioned queries: 0
Missing profiles: 0
Missing synthetic queries: 0

Profile-conditioned query length statistics:
count    5559.000000
mean      224.997122
std        42.571985
min       106.000000
25%       194.000000
50%       224.000000
75%       256.000000
max       352.000000
Name: profile_conditioned_query, dtype: float64


## 6. Encode Profile-Conditioned Queries

Each profile-conditioned text is encoded once using E5-base-v2.

The `query:` prefix follows the E5 query-encoding convention.

The resulting embeddings are L2-normalized 768-dimensional vectors, which
can be directly compared with the normalized product embeddings in the
shared FAISS index.

In [6]:
def encode_profile_conditioned_queries(
    texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        # E5 query format
        inputs = tokenizer(
            [f"query: {text}" for text in batch_texts],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = e5_model(**inputs)

        # Mean pooling using the attention mask
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_embeddings = (
            outputs.last_hidden_state * attention_mask
        )

        sum_embeddings = masked_embeddings.sum(dim=1)
        sum_mask = attention_mask.sum(dim=1).clamp(min=1)

        embeddings = sum_embeddings / sum_mask

        # L2 normalization
        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.append(
            embeddings.cpu().numpy().astype(np.float32)
        )

        if (start // batch_size) % 10 == 0:
            print(
                f"Encoded {min(start + batch_size, len(texts)):,}"
                f"/{len(texts):,}"
            )

    return np.vstack(all_embeddings)


profile_query_embeddings = encode_profile_conditioned_queries(
    eval_data["profile_conditioned_query"].tolist()
)

print("\nEmbedding shape:", profile_query_embeddings.shape)
print("Data type:", profile_query_embeddings.dtype)

# Check normalization
sample_norms = np.linalg.norm(
    profile_query_embeddings[:10],
    axis=1
)

print("First 10 embedding norms:", sample_norms)
print("Mean norm:", np.linalg.norm(profile_query_embeddings, axis=1).mean())

Encoded 256/5,559
Encoded 2,816/5,559
Encoded 5,376/5,559

Embedding shape: (5559, 768)
Data type: float32
First 10 embedding norms: [1.         0.99999994 0.9999999  1.         1.         1.
 1.         0.99999994 1.         0.99999994]
Mean norm: 1.0


## 7. Load the Shared FAISS Product Index

The proposed method uses exactly the same product representations and
FAISS index as the BLAIR-style baseline.

The index contains E5 embeddings for the complete Amazon Electronics
product catalogue.

Because both query and product embeddings are L2-normalized, the
`IndexFlatIP` inner product corresponds to cosine similarity.

In [7]:
# Load the shared FAISS index
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

print("FAISS index loaded successfully.")
print("Index type:", type(faiss_index).__name__)
print("Number of products:", faiss_index.ntotal)
print("Embedding dimension:", faiss_index.d)

# Validate against the expected full catalogue size
assert faiss_index.ntotal == len(pd.read_parquet(
    PRODUCT_CATALOGUE_PATH,
    columns=["parent_asin"]
)), "FAISS index size does not match product catalogue."

assert faiss_index.d == 768, "Unexpected embedding dimension."

print("\nValidation passed.")

FAISS index loaded successfully.
Index type: IndexFlatIP
Number of products: 1610012
Embedding dimension: 768

Validation passed.


## 8. Profile-Conditioned Semantic Retrieval

The profile-conditioned query embeddings are searched against the shared
FAISS index containing the full Amazon Electronics product catalogue.

For each evaluation example, we retrieve the top 50 products ranked by
cosine similarity.

The held-out ground-truth product is not used during retrieval.

In [8]:
# Search the full product catalogue
retrieval_scores, retrieval_indices = faiss_index.search(
    profile_query_embeddings,
    TOP_K
)

print("Retrieval scores shape:", retrieval_scores.shape)
print("Retrieved indices shape:", retrieval_indices.shape)

print("\nExpected shape:", (len(eval_data), TOP_K))

# Basic validation
assert retrieval_scores.shape == (len(eval_data), TOP_K)
assert retrieval_indices.shape == (len(eval_data), TOP_K)

print("\nTop-10 results for the first evaluation example:")
print("Indices:", retrieval_indices[0][:10])
print("Scores:", retrieval_scores[0][:10])

Retrieval scores shape: (5559, 50)
Retrieved indices shape: (5559, 50)

Expected shape: (5559, 50)

Top-10 results for the first evaluation example:
Indices: [ 153548 1026746  812359  606280  763756   82864    2795  818651  950472
  648606]
Scores: [0.83006895 0.82965493 0.8293797  0.82778585 0.82775366 0.8275118
 0.8254615  0.8248308  0.8248011  0.82380116]


## 9. Map Retrieved Indices to Product Identifiers

FAISS returns integer positions in the product index. We map these positions
back to the corresponding `parent_asin` values from the full Amazon Electronics
catalogue.

The ordering is preserved: the first ASIN corresponds to the highest-scoring
retrieved product, the second to the next highest, and so on.

In [10]:
# Load only the product identifiers in FAISS order
product_catalogue = pd.read_parquet(
    PRODUCT_CATALOGUE_PATH,
    columns=["parent_asin"]
)

print("Product catalogue shape:", product_catalogue.shape)
print("FAISS index size:", faiss_index.ntotal)

# Verify alignment
assert len(product_catalogue) == faiss_index.ntotal

# Convert FAISS indices to parent ASINs
retrieved_asins = [
    product_catalogue.iloc[indices]["parent_asin"].tolist()
    for indices in retrieval_indices
]

# Store the retrieval results
proposed_retrievals = eval_data[
    [
        "sample_id",
        "user_id",
        "synthetic_query",
        "profile_text",
        "ground_truth_asin"
    ]
].copy()

proposed_retrievals["retrieved_asins"] = retrieved_asins
proposed_retrievals["retrieval_scores"] = retrieval_scores.tolist()

print("\nProposed retrieval dataframe shape:",
      proposed_retrievals.shape)

print("\nFirst example:")
print("Ground truth:", proposed_retrievals.loc[0, "ground_truth_asin"])
print("Top 10 retrieved ASINs:")
print(proposed_retrievals.loc[0, "retrieved_asins"][:10])

Product catalogue shape: (1610012, 1)
FAISS index size: 1610012

Proposed retrieval dataframe shape: (5559, 7)

First example:
Ground truth: B007VKII6A
Top 10 retrieved ASINs:
['B013P27MDW', 'B072Q8X5PX', 'B013P27OHG', 'B013P27OOO', 'B0195Y015K', 'B07PGL52SD', 'B075B5T7L5', 'B00M2UKG64', 'B01C9I5S1Q', 'B00MICIMPI']


## 10. Save Proposed Model Retrieval Results

The complete Top-50 retrieval results are saved separately from the BLAIR-style
baseline results.

Each row corresponds to one evaluation example and contains the ground-truth
product together with the 50 products retrieved by the proposed
profile-conditioned semantic retrieval method.

In [11]:
PROPOSED_RETRIEVAL_PATH = (
    GENERATED_DIR / "proposed_model_retrievals.parquet"
)

proposed_retrievals.to_parquet(
    PROPOSED_RETRIEVAL_PATH,
    index=False
)

print("Saved to:")
print(PROPOSED_RETRIEVAL_PATH)

print("\nFile exists:", PROPOSED_RETRIEVAL_PATH.exists())
print(
    "File size:",
    round(PROPOSED_RETRIEVAL_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

print("\nShape:", proposed_retrievals.shape)

Saved to:
/users/adhp511/query-conditioned-recommender/data/generated/proposed_model_retrievals.parquet

File exists: True
File size: 4.7 MB

Shape: (5559, 7)


## 11. Qualitative Inspection of Retrieved Products

We inspect a small number of evaluation examples to verify that the
profile-conditioned retrieval produces semantically relevant products.

This provides a qualitative check before proceeding to quantitative
evaluation in the evaluation notebook.

In [12]:
# Load product titles for qualitative inspection
product_titles = pd.read_parquet(
    PRODUCT_CATALOGUE_PATH,
    columns=["parent_asin", "title"]
)

# Create a fast ASIN -> title lookup
title_lookup = dict(
    zip(product_titles["parent_asin"], product_titles["title"])
)

# Inspect the first 5 evaluation examples
for row_idx in range(5):
    row = proposed_retrievals.iloc[row_idx]

    print("=" * 100)
    print(f"Example {row_idx + 1}")
    print(f"Sample ID: {row['sample_id']}")
    print(f"\nQuery: {row['synthetic_query']}")
    print(f"\nGround-truth ASIN: {row['ground_truth_asin']}")
    print(
        "Ground-truth title:",
        title_lookup.get(row["ground_truth_asin"], "NOT FOUND")
    )

    print("\nTop 10 retrieved products:")
    
    for rank, (asin, score) in enumerate(
        zip(
            row["retrieved_asins"][:10],
            row["retrieval_scores"][:10]
        ),
        start=1
    ):
        title = title_lookup.get(asin, "TITLE NOT FOUND")
        print(f"{rank:2d}. [{score:.4f}] {title}")

Example 1
Sample ID: 0

Query: Looking for a high-speed micro sd card for my action camera

Ground-truth ASIN: B007VKII6A
Ground-truth title: IDANCE CRAZY101 - Watt - Channel Recording Studio Equipment, White and Pink

Top 10 retrieved products:
 1. [0.8301] Patriot LX Series 16GB Micro SDHC - Class 10 UHS-I - 5 Pack (PSF16GMCSDHC5PK)
 2. [0.8297] Patriot LX Series 128GB Micro SDXC - Class 10 UHS-I - 5 Pack
 3. [0.8294] Patriot LX Series 8GB Micro SDHC - Class 10 UHS-I - 5 Pack (PSF8GMCSDHC5PK)
 4. [0.8278] Patriot LX Series 32GB Micro SDHC - Class 10 UHS-I - 5 Pack (PSF32GMCSDHC5PK)
 5. [0.8278] Patriot LX Series 128GB Micro SDXC - Class 10 UHS-I - 5 Pack (PSF128GMSXC5PK)
 6. [0.8275] Patriot LX Series 128GB High Speed Micro SDXC Class 10 UHS-I, 2-Pack
 7. [0.8255] Patriot LX Series 64GB High Speed Micro SDXC Class 10 UHS-I Transfer Speeds For Action Cameras, Phones, Tablets, and PCs
 8. [0.8248] Patriot Extreme Performance Series 64 GB MicroSDXC Card U3, UHS-I, Class-10 Compliant - S